<a href="https://colab.research.google.com/github/nanda75/RAG-Notebook/blob/main/assignment_1_multimodal_vector_db.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1: Vector Database Creation and Retrieval
## Day 7 - RAG Fundamentals

The goal is not to build a completely new system from scratch. The goal is to repeat the same flow with guided TODOs so the first notebook feels intuitive.

## What you will build

You will build a small multimodal RAG retrieval pipeline using LlamaIndex and LanceDB:

1. Mount Google Drive and install dependencies
2. Configure LlamaIndex settings
3. Explore a folder containing different file types
4. Load documents using `SimpleDirectoryReader`
5. Create a LanceDB vector store
6. Create a `StorageContext`
7. Build a `VectorStoreIndex`
8. Retrieve relevant chunks for a query
9. Inspect retrieved chunks and metadata
10. Optionally generate an answer using a query engine

## Main idea

Yesterday's session and notebooks showed the full RAG system end-to-end. This assignment asks you to complete the similar flow step by step.

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it

## 0. Mount Google Drive

Use this if you are running the notebook in Google Colab.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Install dependencies

Use the same requirements file path used in class. If your folder path is different, update it below.

In [4]:
!pip install -q -r "/content/drive/MyDrive/Outskill/requirements.txt" # this path will change for you
!pip install -q gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.9/369.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 5.9 MB/s eta 0:

In [5]:
!apt-get update && apt-get install -y ffmpeg
!pip install openai-whisper

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,179 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,260 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,965 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:14 http://security.ubu

## 2. Set OpenRouter API key

We use OpenRouter for the LLM, similar to the class notebook. The embedding model will be local.

In [6]:
import os
from google.colab import userdata

# securely input your key
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
print("✓ OpenRouter key set successfully")

✓ OpenRouter key set successfully


## 3. Imports and configuration

Before running the full pipeline, check these values:

- `data_path`: folder that contains your input files (CHANGE IT ACCORDING TO YOUR OWN PATH)
- `vector_db_path`: where LanceDB will store vectors
- `index_storage_path`: where LlamaIndex will persist index information
- `chunk_size` and `chunk_overlap`: same concepts discussed in class

In [7]:
# Import required libraries
import os
from pathlib import Path
import time
from typing import List

print("Libraries imported successfully!")

Libraries imported successfully!


In [8]:
CONFIG = {
    "llm_model": "gpt-5-mini",
    "embedding_model": "local:BAAI/bge-small-en-v1.5",
    "chunk_size": 512,
    "chunk_overlap": 50,
    "similarity_top_k": 5,
    "data_path": "/content/drive/MyDrive/Outskill/data",  # change the path accordingly
    "vector_db_path": "/content/storage/assignment_multimodal_vectordb",
    "index_storage_path": "/content/storage/assignment_multimodal_index",
    "table_name": "assignment_documents"
}

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Configuration loaded")
print("Data path:", CONFIG["data_path"])
print("Vector DB path:", CONFIG["vector_db_path"])
print("Index storage path:", CONFIG["index_storage_path"])

Configuration loaded
Data path: /content/drive/MyDrive/Outskill/data
Vector DB path: /content/storage/assignment_multimodal_vectordb
Index storage path: /content/storage/assignment_multimodal_index


## 4. Configure LlamaIndex settings

This step defines:

- which LLM will generate answers
- which embedding model will convert text into vectors
- how text will be chunked

### Your task

Complete the function below.

Hint: This is very similar to the `configure_llamaindex_settings()` function from the class notebook.

In [9]:
from llama_index.core import Settings
from llama_index.llms.openrouter import OpenRouter
from llama_index.core.embeddings import resolve_embed_model
from llama_index.core.node_parser import SentenceSplitter

def configure_llamaindex_settings():
    """Configure LlamaIndex global settings using hardcoded configuration."""

    # Set up LLM with OpenRouter using hardcoded model
    Settings.llm = OpenRouter(
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model=CONFIG["llm_model"]
    )
    print(f"✓ LLM configured: {CONFIG['llm_model']}")

    # Set up local embedding model (downloads locally first time, then cached)
    Settings.embed_model = resolve_embed_model(CONFIG["embedding_model"])
    print(f"✓ Embedding model configured: {CONFIG['embedding_model']}")

    # Set up node parser for chunking with hardcoded settings
    Settings.node_parser = SentenceSplitter(
        chunk_size=CONFIG["chunk_size"],
        chunk_overlap=CONFIG["chunk_overlap"]
    )
    print(f"✓ Text chunking configured: {CONFIG['chunk_size']} chars with {CONFIG['chunk_overlap']} overlap")

# Configure the settings
configure_llamaindex_settings()
print("✓ LlamaIndex settings configured for multimodal processing")

✓ LLM configured: gpt-5-mini


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Embedding model configured: local:BAAI/bge-small-en-v1.5
✓ Text chunking configured: 512 chars with 50 overlap
✓ LlamaIndex settings configured for multimodal processing


## 5. Explore the dataset

Before loading documents, always inspect what files are present.

This helps you answer questions like:

- How many files are present?
- What file types are present?
- Are we working with PDFs only or multiple formats?
- Is the folder path correct?

In [10]:
def explore_dataset(data_path: str = None):
    """
    Explore and categorize the files in our dataset by type.

    Args:
        data_path (str): Path to the data directory
    """
    if data_path is None:
        data_path = CONFIG["data_path"]

    data_dir = Path(data_path)
    if not data_dir.exists():
        print(f"Data directory not found: {data_dir}")
        return

    # Categorize files by type
    file_types = {}
    all_files = []

    # Walk through all files recursively
    for file_path in data_dir.rglob("*"):
        if file_path.is_file():
            suffix = file_path.suffix.lower()
            file_size = file_path.stat().st_size

            if suffix not in file_types:
                file_types[suffix] = []

            file_info = {
                "path": str(file_path),
                "name": file_path.name,
                "size_mb": round(file_size / (1024 * 1024), 2),
                "size_bytes": file_size
            }

            file_types[suffix].append(file_info)
            all_files.append(file_info)

    # Display summary
    print("---Dataset Overview---")
    print(f"Total files found: {len(all_files)}")

    print(f"\nFile Types Distribution:")
    for file_type, files in sorted(file_types.items()):
        if file_type:  # Skip files without extension
            total_size = sum(f["size_mb"] for f in files)
            print(f"  {file_type}: {len(files)} files ({total_size:.2f} MB)")

            # Show file details
            for file_info in files[:3]:  # Show first 3 files of each type
                print(f"    - {file_info['name']} ({file_info['size_mb']} MB)")
            if len(files) > 3:
                print(f"    ... and {len(files) - 3} more")

            print()

    return file_types, all_files

# Explore our dataset
file_types, all_files = explore_dataset()
print(f"✓ Found {len(all_files)} files across {len(file_types)} different file types")


---Dataset Overview---
Total files found: 21

File Types Distribution:
  .csv: 4 files (0.00 MB)
    - investment_portfolio.csv (0.0 MB)
    - agent_evaluation_metrics.csv (0.0 MB)
    - agent_performance_benchmark.csv (0.0 MB)
    ... and 1 more

  .html: 2 files (0.00 MB)
    - agent_tutorial.html (0.0 MB)
    - fitness_tracker.html (0.0 MB)

  .md: 4 files (0.00 MB)
    - city_guides.md (0.0 MB)
    - market_analysis.md (0.0 MB)
    - agent_framework_comparison.md (0.0 MB)
    ... and 1 more

  .mp3: 3 files (2.95 MB)
    - ai_agents.mp3 (1.54 MB)
    - rags.mp3 (0.81 MB)
    - in_the_end.mp3 (0.6 MB)

  .pdf: 2 files (1.92 MB)
    - Emerging_Agent_Architectures.pdf (1.58 MB)
    - AI_Agent_Frameworks.pdf (0.34 MB)

  .png: 6 files (0.55 MB)
    - agent_performance_comparison.png (0.17 MB)
    - agent_types_comparison.png (0.1 MB)
    - fitness_progress.png (0.08 MB)
    ... and 3 more

✓ Found 21 files across 6 different file types


## 6. Load multimodal documents

In the class notebook, we used `SimpleDirectoryReader` because it can load many file types such as PDF, CSV, Markdown, HTML, images, and notebooks.

At this stage, files become LlamaIndex `Document` objects.

### Your task

Complete the function below to load documents from the dataset folder.

In [11]:
from llama_index.core import SimpleDirectoryReader

def load_documents_from_folder(data_path: str= None, recursive: bool = True):
    """
    Load documents from multiple file types using SimpleDirectoryReader.

    Args:
        data_path (str): Path to directory containing multimodal data
        recursive (bool): Whether to search subdirectories

    Returns:
        List of Document objects
    """
    if data_path is None:
      data_path = CONFIG["data_path"]

    print(f"📂 Loading multimodal documents from: {data_path}")

    # Create SimpleDirectoryReader with recursive search
    reader = SimpleDirectoryReader(
        input_dir=data_path,
        recursive=recursive,
        # Let SimpleDirectoryReader handle all supported file types automatically
    )

    print("🔄 Processing files...")
    start_time = time.time()

    # Load all documents
    documents = reader.load_data()

    end_time = time.time()

    print(f"✅ Successfully loaded {len(documents)} documents in {end_time - start_time:.2f} seconds")

    # Analyze loaded documents by file type
    doc_types = {}
    for doc in documents:
        file_type = doc.metadata.get('file_type', 'unknown')
        if file_type not in doc_types:
            doc_types[file_type] = []
        doc_types[file_type].append(doc)

    print(f"\n📊 Documents by MIME type:")
    for mime_type, docs in sorted(doc_types.items()):
        print(f"  {mime_type}: {len(docs)} documents")

    return documents


# Test the function after you complete it
test_folder = CONFIG["data_path"]
documents = load_documents_from_folder(test_folder)
print(f"Loaded {len(documents)} documents")

📂 Loading multimodal documents from: /content/drive/MyDrive/Outskill/data
🔄 Processing files...


100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 97.5MiB/s]
/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


✅ Successfully loaded 42 documents in 52.99 seconds

📊 Documents by MIME type:
  application/pdf: 23 documents
  audio/mpeg: 3 documents
  image/png: 6 documents
  text/csv: 4 documents
  text/html: 2 documents
  text/markdown: 4 documents
Loaded 42 documents


## 7. Create LanceDB vector store

The vector store is where embeddings are stored and searched.

Important idea:

- LlamaIndex creates chunks and embeddings
- LanceDB stores the vector representation
- Later, the retriever searches this vector store

### Your task

Complete the function below.

In [12]:
import lancedb
# Vector store and index creation
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

def create_vector_store(vector_db_path: str, table_name: str):
    """
    Create a LanceDB vector store for storing document embeddings.

    TODO: In this function, you need to:
    1. Create the database directory if it does not already exist.
    2. Create a LanceDBVectorStore object.
    3. Return the vector store.

    Args:
        db_path (str): Path where the vector database will be stored.
        table_name (str): Name of the table in the vector database.

    Returns:
        LanceDBVectorStore: Configured vector store.
    """
    try :
      # TODO: Create the directory if it doesn't exist
      Path(vector_db_path).mkdir(parents=True, exist_ok=True)

      # TODO: Connect to LanceDB (creates a connection to the LanceDB database)
      db = lancedb.connect(str(vector_db_path))
      print(f"✓ Connected to LanceDB at: {vector_db_path}")

      # TODO: Create vector store (HINT: Use LanceDBVectorStore)
      # vector_store = ?

      vector_store = LanceDBVectorStore(
              uri=str(vector_db_path),
              table_name="multimodal_documents"
          )
      print("✓ LanceDB vector store created for multimodal data")

      return vector_store

      # return vector_store
    except Exception as e:
      print(f"Error creating LanceDB vector store: {e}")
      return None


# Test the function after you complete it
vector_store = create_vector_store(vector_db_path=CONFIG["vector_db_path"], table_name=CONFIG["table_name"])
print(f"Vector store created: {vector_store is not None}")

✓ Connected to LanceDB at: /content/storage/assignment_multimodal_vectordb
✓ LanceDB vector store created for multimodal data
Vector store created: True


## 8. Create StorageContext and VectorStoreIndex

This is one of the most important parts of the assignment.

### What is `StorageContext`?

Think of it as the object that tells LlamaIndex where the prepared index data should live.

In this assignment:

- `vector_store` stores embeddings
- `StorageContext` connects LlamaIndex to that vector store
- `VectorStoreIndex.from_documents()` builds the index from documents

### Your task

Complete the function below.

In [23]:
def create_vector_index(documents: List, vector_store, persist_dir: str = None):
    """
    Create a VectorStoreIndex from loaded documents.

    Args:
        documents: Loaded LlamaIndex documents.
        vector_store: LanceDB vector store.
        persist_dir: Folder for persisting index metadata.

    Returns:
        VectorStoreIndex object.
    """
    if persist_dir is None:
        persist_dir = CONFIG["index_storage_path"]

    if not documents:
        raise ValueError("No documents found. Load documents before creating the index.")

    persist_dir_path = Path(persist_dir)
    persist_dir_path.mkdir(parents=True, exist_ok=True)

     # Check if index already exists
    index_store_file = persist_dir_path / "index_store.json"

    if index_store_file.exists():

      print("Creating storage context")
      # TODO: Create storage context from vector_store (HINT: Use StorageContext)
      print("📁 Loading existing multimodal index...")
      storage_context = StorageContext.from_defaults(
            persist_dir=persist_dir,
            vector_store=vector_store
        )
      # TODO: Create the VectorStoreIndex from documents
      index = VectorStoreIndex.from_vector_store(
            vector_store=vector_store,
            storage_context=storage_context
        )
      print("✓ Successfully loaded existing multimodal index")
      return index
    if not documents:
        raise ValueError("No documents found. Load documents before creating the index.")
    print("🔨 Creating new vector store index...")
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context
    )
    # Persist the new index metadata to disk
    print("💾 Saving multimodal index to storage...")
    index.storage_context.persist(persist_dir=persist_dir)
    print("✓ Successfully saved new multimodal index")

    return index
if vector_store and documents:
    index = create_vector_index(documents, vector_store)

🔨 Creating new vector store index...
💾 Saving multimodal index to storage...
✓ Successfully saved new multimodal index


## 9. Create a retriever
A retriever does not generate an answer. It only returns the most relevant chunks.

This is useful because it lets you inspect what the RAG system found before the LLM answers.

### Your task

Complete the retriever setup.

In [24]:
from llama_index.core.retrievers import VectorIndexRetriever

def create_retriever(index, similarity_top_k: int = None):
    """
    Create a retriever from the index.

    Args:
        index: VectorStoreIndex object.
        similarity_top_k: Number of chunks to retrieve.

    Returns:
        VectorIndexRetriever object.
    """
    if similarity_top_k is None:
        similarity_top_k = CONFIG["similarity_top_k"]
    if not index:
        print("❌ Index not available. Please create index first.")
        return None
    try:
      # TODO: Create VectorIndexRetriever
      retriever = VectorIndexRetriever(
            index=index,
            similarity_top_k=similarity_top_k,
      )
      print(f"Retriever created with similarity_top_k={similarity_top_k}")
      return retriever
    except Exception as e:
      print(f"Error creating retriever: {e}")
      return None
retriever = create_retriever(index)

Retriever created with similarity_top_k=5


## 10. Retrieve chunks for a query

This step helps you see exactly what semantic search returns.

Good queries to try:

- Ask about a topic you know exists in the dataset
- Ask using different wording than the original document
- Ask a broad question and inspect whether results are noisy

In [25]:
def retrieve_chunks(retriever, query: str, show_text: bool = True):
    """
    Retrieve relevant chunks for a query and print their metadata and score.

    Args:
        retriever: VectorIndexRetriever object.
        query: User query.
        show_text: Whether to print text previews.

    Returns:
        Retrieved nodes.
    """
    print("Query:", query)
    print("=" * 80)

    nodes = retriever.retrieve(query)

    print("Retrieved chunks:", len(nodes))

    for i, node_with_score in enumerate(nodes, 1):
        node = node_with_score.node
        score = node_with_score.score
        metadata = node.metadata

        print(f"Result {i}")
        print("Score:", score)
        print("File name:", metadata.get("file_name", "unknown"))
        print("File type:", metadata.get("file_type", "unknown"))

        if show_text:
            print("Text preview:")
            print(node.get_content()[:700])

        print("-" * 80)

    return nodes

sample_query = "Where should I travel next in May-June?"
retrieved_nodes = retrieve_chunks(retriever, sample_query)

Query: Where should I travel next in May-June?
Retrieved chunks: 5
Result 1
Score: 0.5675950050354004
File name: city_guides.md
File type: text/markdown
Text preview:
# Ultimate City Travel Guide

## Paris, France 🇫🇷

**Best Time to Visit:** April-June, September-October
**Must-See Attractions:**
- Eiffel Tower - Iconic iron lattice tower
- Louvre Museum - World's largest art museum
- Notre-Dame Cathedral - Gothic masterpiece
- Champs-Élysées - Famous shopping avenue

**Local Cuisine:** Croissants, Escargot, Coq au Vin, Macarons
**Transportation:** Metro system, Vélib bike sharing
**Budget:** €100-150 per day for mid-range travel

---

## Tokyo, Japan 🇯🇵

**Best Time to Visit:** March-May (cherry blossoms), September-November
**Must-See Attractions:**
- Senso-ji Temple - Ancient Buddhist temple
- Shibuya Crossing - World's busiest pedestrian crossing
- To
--------------------------------------------------------------------------------
Result 2
Score: 0.36790528893470764
File name: city

## 11. Build a query engine

A retriever only returns chunks. A query engine uses the retriever and an LLM to generate a final response.

This mirrors the class notebook section where we created a multimodal query engine.

### Your task

Complete the function below.

In [26]:
from llama_index.core.query_engine import RetrieverQueryEngine

def create_query_engine(retriever, similarity_top_k: int = None):
    """
    Create a RetrieverQueryEngine using a retriever.

    Args:
        retriever: VectorIndexRetriever object.
        similarity_top_k: Number of chunks to retrieve.

    Returns:
        RetrieverQueryEngine object.
    """
    if similarity_top_k is None:
        similarity_top_k = CONFIG["similarity_top_k"]
    try:
        # TODO: Create query engine (HINT: Use RetrieverQueryEngine)
        query_engine = RetrieverQueryEngine(retriever=retriever)

        print("Query engine created")
        return query_engine
    except Exception as e:
        print(f"Error creating query engine: {e}")
        return None

query_engine = create_query_engine(retriever)

Query engine created


## 12. Ask a question and inspect sources

This is the final RAG step.

We ask a question, get a generated answer, and then inspect which source chunks were used.

In [35]:
# Check how many nodes/vectors are in your index
import json
import requests


def ask_question_direct_llm(
    retriever, question: str, show_sources: bool = True
):
  print("Question:", question)
  print("=" * 80)

  # 1. Retrieve relevant nodes using LlamaIndex retriever
  source_nodes = retriever.retrieve(question)

  # 2. Build context string from retrieved chunks
  context_str = "\n\n".join(
      [f"Source {i+1}:\n{node.node.get_content()}" for i, node in enumerate(source_nodes)]
  )

  # 3. Create prompt for OpenRouter
  prompt = f"""Context information is below.
---------------------
{context_str}
---------------------
Given the context information and not prior knowledge, answer the query.
Query: {question}
Answer:"""

  # 4. Make direct request to OpenRouter API
  api_key = os.getenv("OPENROUTER_API_KEY")
  url = "https://openrouter.ai/api/v1/chat/completions"

  headers = {
      "Authorization": f"Bearer {api_key}",
      "Content-Type": "application/json",
  }

  payload = {
      "model": "openai/gpt-4o-mini",  # You can also try "anthropic/claude-3-haiku" or "google/gemini-flash-1.5"
      "messages": [{"role": "user", "content": prompt}],
      "temperature": 0.2,
  }

  response = requests.post(url, headers=headers, json=payload)

  print("Answer:")
  if response.status_code == 200:
    res_json = response.json()
    answer = res_json["choices"][0]["message"]["content"]
    print(answer)
  else:
    print(f"Error {response.status_code}: {response.text}")

  # 5. Display retrieved sources
  if show_sources:
    print("\nSources used:")
    for i, source in enumerate(source_nodes, 1):
      node = source.node
      metadata = node.metadata
      print(f"Source {i}")
      print("Score:", source.score)
      print("File name:", metadata.get("file_name", "unknown"))
      print("File type:", metadata.get("file_type", "unknown"))
      print("Text preview:")
      print(node.get_content()[:500])
      print("-" * 80)


# Initialize retriever directly from your index
retriever = index.as_retriever(similarity_top_k=CONFIG["similarity_top_k"])

# Test with your query
question = "Where should I travel in May or June?"
ask_question_direct_llm(retriever, question, show_sources=True)

Question: Where should I travel in May or June?
Answer:
In May or June, you should consider traveling to either Paris, France, or Tokyo, Japan. 

- **Paris, France**: This is one of the best times to visit, with pleasant weather and vibrant spring activities. You can explore iconic attractions like the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral, and enjoy local cuisine such as croissants and escargot.

- **Tokyo, Japan**: May is also a great time to visit Tokyo, especially to see the cherry blossoms in bloom. Must-see attractions include Senso-ji Temple, Shibuya Crossing, and Tokyo Skytree, along with delicious local cuisine like sushi and ramen.

Both cities offer unique experiences and are ideal destinations during these months.

Sources used:
Source 1
Score: 0.5612598657608032
File name: city_guides.md
File type: text/markdown
Text preview:
# Ultimate City Travel Guide

## Paris, France 🇫🇷

**Best Time to Visit:** April-June, September-October
**Must-See Attractions:**
- 

In [34]:
def ask_question(query_engine, question: str, show_sources: bool = True):
    """
    Ask a question to the query engine and display answer plus sources.

    Args:
        query_engine: RetrieverQueryEngine object.
        question: User question.
        show_sources: Whether to show source chunks.
    """
    print("Question:", question)
    print("=" * 80)

    response = query_engine.query(question)

    print("Answer:")
    print(str(response))

    if show_sources:
        print("Sources used:")
        source_nodes = getattr(response, "source_nodes", [])
        for i, source in enumerate(source_nodes, 1):
            node = source.node
            metadata = node.metadata
            print(f"Source {i}")
            print("Score:", source.score)
            print("File name:", metadata.get("file_name", "unknown"))
            print("File type:", metadata.get("file_type", "unknown"))
            print("Text preview:")
            print(node.get_content()[:500].replace("", " "))
            print("-" * 80)

# Try your own question here
question = "What are the steps to make Carbonara?"
ask_question(query_engine, question, show_sources=True)

Question: Where should I travel in May or June?
Answer:
Empty Response
Sources used:
Source 1
Score: 0.5612598657608032
File name: city_guides.md
File type: text/markdown
Text preview:
# Ultimate City Travel Guide

## Paris, France 🇫🇷

**Best Time to Visit:** April-June, September-October
**Must-See Attractions:**
- Eiffel Tower - Iconic iron lattice tower
- Louvre Museum - World's largest art museum
- Notre-Dame Cathedral - Gothic masterpiece
- Champs-Élysées - Famous shopping avenue

**Local Cuisine:** Croissants, Escargot, Coq au Vin, Macarons
**Transportation:** Metro system, Vélib bike sharing
**Budget:** €100-150 per day for mid-range travel

---

## Tokyo, Japan 🇯🇵

**B
--------------------------------------------------------------------------------
Source 2
Score: 0.3614733815193176
File name: city_temperatures.png
File type: image/png
Text preview:

--------------------------------------------------------------------------------
Source 3
Score: 0.3513989746570587
File name: ma

## 13. Launch the RAG app with Gradio

Now that the complete RAG pipeline is ready, use the interface below to ask questions without calling Python functions manually.

The app displays both the generated answer and the retrieved source chunks so that you can verify where the answer came from. Run this cell only after `query_engine` has been created successfully.

When running in Google Colab, `share=True` creates a temporary public link. Do not enter private or sensitive information into a publicly shared app.

In [42]:
import json
import os
import gradio as gr
import requests


def ask_question_direct_llm(question: str):
  if not question.strip():
    return "Please enter a question.", ""

  # 1. Retrieve relevant nodes using your LlamaIndex retriever
  source_nodes = retriever.retrieve(question)

  # 2. Build context string from retrieved chunks
  context_str = "\n\n".join([
      f"Source {i+1}:\n{node.node.get_content()}"
      for i, node in enumerate(source_nodes)
  ])

  # 3. Construct prompt
  prompt = f"""Context information is below.
---------------------
{context_str}
---------------------
Given the context information and not prior knowledge, answer the query.
Query: {question}
Answer:"""

  # 4. Direct API call to OpenRouter
  api_key = os.getenv("OPENROUTER_API_KEY")
  url = "https://openrouter.ai/api/v1/chat/completions"

  headers = {
      "Authorization": f"Bearer {api_key}",
      "Content-Type": "application/json",
  }

  payload = {
      "model": "openai/gpt-4o-mini",  # or your preferred OpenRouter model ID
      "messages": [{"role": "user", "content": prompt}],
      "temperature": 0.2,
  }

  try:
    api_response = requests.post(url, headers=headers, json=payload)

    if api_response.status_code == 200:
      res_json = api_response.json()
      answer = res_json["choices"][0]["message"]["content"]
    else:
      answer = f"Error {api_response.status_code}: {api_response.text}"
  except Exception as e:
    answer = f"API Request Failed: {str(e)}"

  # 5. Format retrieved source nodes for the UI
  source_sections = []
  for i, source in enumerate(source_nodes, 1):
    node = source.node
    metadata = node.metadata or {}
    score = (
        "unknown"
        if getattr(source, "score", None) is None
        else f"{source.score:.4f}"
    )
    preview = node.get_content()[:500].replace("\n", " ")

    source_sections.append(
        f"Source {i}\n"
        f"File: {metadata.get('file_name', 'unknown')}\n"
        f"Type: {metadata.get('file_type', 'unknown')}\n"
        f"Score: {score}\n"
        f"Preview: {preview}\n"
    )

  sources_formatted = "\n\n".join(
      source_sections
  ) or "No source nodes were returned."

  return answer, sources_formatted

demo = gr.Interface(
    fn=ask_question_direct_llm,
    inputs=gr.Textbox(
        label="Question",
        placeholder="Ask a question about your documents",
        lines=2,
    ),
    outputs=[
        gr.Textbox(label="Answer", lines=8),
        gr.Textbox(label="Retrieved sources", lines=14),
    ],
    title="Multimodal RAG Question Answering",
    description="Ask questions about the files indexed in your LanceDB vector store.",
    examples=[
        ["What are the steps to make Carbonara?"],
        ["Where should I travel in May or June?"],
        ["Summarize the most relevant information in the dataset."],
    ],
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/08/30 10:04:29 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>

## Conclusion

🎉 **Congratulations!** You have successfully built an advanced **Multimodal RAG System** using LlamaIndex's `SimpleDirectoryReader` with comprehensive cross-modal capabilities.

## Student experiments (for trying out later after the session)

Complete these small experiments later to build intuition.

### Experiment 1: Change `similarity_top_k`

Try values like 2, 5, and 10.

Question to answer:

- Does increasing `top_k` always improve the answer?
- Do you see more noise when `top_k` is high?

### Experiment 2: Ask the same question in different wording

Example:

- "What is the refund policy?"
- "Can customers get their money back?"

Question to answer:

- Does semantic search still retrieve similar chunks?

### Experiment 3: Inspect sources before trusting the answer

Question to answer:

- Did the LLM answer using the right sources?
- Were any irrelevant chunks included?

In [ ]:
# Experiment area
# Change the query and top_k values below.

experiment_query = "Replace this with your own question"
experiment_top_k = 3

experiment_retriever = create_retriever(index, similarity_top_k=experiment_top_k)
experiment_nodes = retrieve_chunks(experiment_retriever, experiment_query)

## Reflection questions

Answer these in a markdown cell below after you complete the notebook.

1. Why do we parse and load files before indexing?
2. Why do we chunk text instead of storing the full document as one unit?
3. What does `chunk_overlap` help with?
4. What is stored in the vector database?
5. What role does `StorageContext` play?
6. What is the difference between retriever output and query engine output?
7. Give one example where returning retrieved chunks directly is better than generating an answer.
8. Give one example where generation is useful after retrieval.

In [ ]:
# Write short answers here as comments or create a markdown cell below.

# 1.
# 2.
# 3.
# 4.
# 5.
# 6.
# 7.
# 8.